# IAN 603 · Module 3
# PostgreSQL in Action

This notebook is a practical introduction to working with PostgreSQL.

The goal is not to memorize isolated commands. Instead, we will build a repeatable workflow:

```text
Ask a question
      ↓
Read the data
      ↓
Filter / calculate / sort
      ↓
Inspect the result
      ↓
Change data only when needed
      ↓
Verify the change
```

By the end, you should be comfortable reading data, making safe changes, using transactions, working with JSONB, and understanding when PostgreSQL features such as `MERGE` are useful.

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module3.html


## What you will learn

By the end of this notebook, you should be able to:

1. Write a clear `SELECT` statement.
2. Choose columns instead of automatically using `SELECT *`.
3. Create aliases and calculated columns.
4. Filter with `WHERE`, `AND`, `BETWEEN`, `IN`, and `ILIKE`.
5. Handle missing values correctly with `IS NULL`.
6. Sort, limit, and remove duplicate values.
7. Explain the logical flow of a query.
8. Use `INSERT`, `UPDATE`, and `DELETE` safely.
9. Use `RETURNING` to inspect changed rows.
10. Use transactions to test a change before keeping it.
11. Extract values from PostgreSQL `JSONB`.
12. Build a table from query results.
13. Understand the purpose of `MERGE`.


## 0. Install required Python packages

Run this cell if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** When you run the next cell, the helper is downloaded automatically from the course GitHub repository if it is not already available.

If you want your own copy of the helper:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)


In [ ]:
from pathlib import Path
import urllib.request
import pandas as pd
from IPython.display import display
from sqlalchemy import text

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link above, then run this cell again.")
        raise

from ian603_database import IAN603Database

db = IAN603Database()

display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script

print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window.

The usual local PostgreSQL defaults are already filled in:

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password and click **Connect**.


In [ ]:
db.show_connection_panel()


## 3. Check the connection

After clicking **Connect**, run this cell once.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


# Part I · Meet the Teaching Database

We will use a small clinical-style database with four tables.

```text
patients
   │
   ├────────< admissions >──────── doctors
   │
   └────────< clinical_notes
```

### Row grain

| Table | One row represents |
|---|---|
| `patients` | one patient |
| `doctors` | one doctor |
| `admissions` | one admission |
| `clinical_notes` | one JSON clinical note |

Understanding the grain first makes later queries easier to reason about.


## 4. Create or reset the Module 3 database

You can return to this cell at any time if you want to restore the original teaching data.


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 3\n-- PostgreSQL in Action\n-- ============================================================\n-- Companion file for s_module3.html\n-- PostgreSQL / pgAdmin Query Tool\n--\n-- Topics:\n-- SELECT, filtering, aliases, INSERT, UPDATE, DELETE, RETURNING,\n-- transactions, JSONB, CREATE TABLE AS, and MERGE.\n--\n-- SAFETY HABIT\n-- Before UPDATE or DELETE, preview the target rows with SELECT.\n-- Use RETURNING when useful, and use transactions when testing.\n-- ============================================================\n\n-- ============================================================\n-- SECTION 1 - CREATE / RESET TEACHING DATABASE\n-- ============================================================\n\nDROP TABLE IF EXISTS clinical_notes, admissions, doctors, patients CASCADE;\n    CREATE TABLE patients (\n      patient_id integer GENERATED ALWAYS AS IDENTITY PRIMARY KEY,\n      mrn text NOT NULL UNIQUE,\n      full_name text NOT NULL,\n      age smallint NOT NULL CHECK (age BETWEEN 0 AND 120),\n      gender text NOT NULL,\n      is_active boolean NOT NULL DEFAULT TRUE\n    );\n    CREATE TABLE doctors (\n      doctor_id integer GENERATED ALWAYS AS IDENTITY PRIMARY KEY,\n      full_name text NOT NULL,\n      specialty text NOT NULL\n    );\n    CREATE TABLE admissions (\n      admission_id integer GENERATED ALWAYS AS IDENTITY PRIMARY KEY,\n      patient_id integer NOT NULL REFERENCES patients(patient_id) ON DELETE RESTRICT,\n      doctor_id integer NOT NULL REFERENCES doctors(doctor_id) ON DELETE RESTRICT,\n      diagnosis text NOT NULL,\n      admission_date date NOT NULL,\n      discharge_date date,\n      status text NOT NULL DEFAULT 'Admitted',\n      CHECK (discharge_date IS NULL OR discharge_date >= admission_date)\n    );\n    CREATE TABLE clinical_notes (\n      note_id integer GENERATED ALWAYS AS IDENTITY PRIMARY KEY,\n      patient_id integer NOT NULL REFERENCES patients(patient_id) ON DELETE CASCADE,\n      note jsonb NOT NULL\n    );\n    INSERT INTO patients (mrn,full_name,age,gender,is_active) VALUES\n      ('MRN1001','Carol Nguyen',42,'Female',TRUE),\n      ('MRN1002','Bob Smith',68,'Male',TRUE),\n      ('MRN1003','Elena Garcia',34,'Female',TRUE),\n      ('MRN1004','David Lee',72,'Male',TRUE),\n      ('MRN1005','Amina Yusuf',58,'Female',TRUE),\n      ('MRN1006','Marcus Reed',27,'Male',TRUE);\n    INSERT INTO doctors (full_name,specialty) VALUES\n      ('Jordan Chen','Family Medicine'),\n      ('Priya Shah','Internal Medicine'),\n      ('Noah Bennett','Cardiology');\n    INSERT INTO admissions (patient_id,doctor_id,diagnosis,admission_date,discharge_date,status) VALUES\n      (1,1,'Asthma','2023-02-18','2023-02-20','Discharged'),\n      (2,3,'Chest pain','2023-03-04','2023-03-07','Discharged'),\n      (4,2,'Dehydration','2023-03-10',NULL,'Admitted'),\n      (5,1,'Migraine','2023-04-02','2023-04-03','Discharged'),\n      (3,2,'Pneumonia','2023-05-11',NULL,'Admitted');\n    INSERT INTO clinical_notes (patient_id,note) VALUES\n      (1,'{\"symptom\":\"wheezing\",\"severity\":\"moderate\",\"follow_up\":true}'::jsonb),\n      (2,'{\"symptom\":\"chest pain\",\"severity\":\"severe\",\"follow_up\":true}'::jsonb),\n      (3,'{\"symptom\":\"cough\",\"severity\":\"moderate\",\"follow_up\":true}'::jsonb),\n      (5,'{\"symptom\":\"headache\",\"severity\":\"mild\",\"follow_up\":false}'::jsonb);"

execute_script(setup_sql)


## 5. Preview the tables

Do not start by writing a complicated query. First, become familiar with the data.


In [ ]:
display_query("""
SELECT patient_id, mrn, full_name, age, gender, is_active
FROM patients
ORDER BY patient_id;
""", title="Patients")


In [ ]:
display_query("""
SELECT admission_id, patient_id, doctor_id, diagnosis,
       admission_date, discharge_date, status
FROM admissions
ORDER BY admission_id;
""", title="Admissions")


# Part II · Reading Data with SELECT

The basic pattern is:

```sql
SELECT column1, column2
FROM table_name;
```

Think of it as two questions:

1. **FROM** — where are the rows coming from?
2. **SELECT** — which values should appear in the result?

Although SQL is written with `SELECT` first, it helps to think about the source table before the output columns.


## 6. Choose only the columns you need

Compare these ideas:

```sql
SELECT *
FROM patients;
```

versus:

```sql
SELECT mrn, full_name, age
FROM patients;
```

For quick exploration, `SELECT *` can be useful. For analytical work, explicit columns usually make the query clearer and more stable.


In [ ]:
display_query("""
SELECT mrn, full_name, age
FROM patients
ORDER BY age DESC;
""", title="Selected patient columns")


## 7. Aliases and calculated columns

A query can return values that are not physically stored as columns.

Suppose the source row is:

| mrn | full_name | age |
|---|---|---:|
| MRN1002 | Bob Smith | 68 |

We can calculate:

```sql
age >= 65 AS is_senior
```

and return:

| mrn | patient | age | is_senior |
|---|---|---:|---|
| MRN1002 | Bob Smith | 68 | true |

The source table is unchanged. The calculation exists only in the query result.


In [ ]:
display_query("""
SELECT
    mrn,
    full_name AS patient,
    age,
    age >= 65 AS is_senior
FROM patients
ORDER BY age DESC;
""", title="Aliases and a calculated Boolean column")


# Part III · Filtering Rows with WHERE

`WHERE` decides which rows continue through the query.

Start with a rule in plain language:

> Keep active patients who are at least 50 years old.

Then translate each part:

| Plain language | SQL |
|---|---|
| active | `is_active = TRUE` |
| at least 50 | `age >= 50` |
| both conditions | `AND` |


## 8. Combine conditions with AND


In [ ]:
display_query("""
SELECT mrn, full_name, age, is_active
FROM patients
WHERE is_active = TRUE
  AND age >= 50
ORDER BY age DESC;
""", title="Active patients age 50 or older")


## 9. BETWEEN and IN

`BETWEEN` is convenient for an inclusive range:

```sql
age BETWEEN 30 AND 60
```

means:

```text
age >= 30 AND age <= 60
```

`IN` is useful when several exact values are acceptable:

```sql
specialty IN ('Cardiology', 'Internal Medicine')
```


In [ ]:
display_query("""
SELECT mrn, full_name, age, gender
FROM patients
WHERE gender = 'Female'
  AND age BETWEEN 30 AND 60
ORDER BY age;
""", title="Female patients between 30 and 60")


## 10. Pattern matching with ILIKE

PostgreSQL `ILIKE` performs case-insensitive pattern matching.

| Pattern | Meaning |
|---|---|
| `'c%'` | begins with c |
| `'%son'` | ends with son |
| `'%ar%'` | contains ar |

The percent sign means **zero or more characters**.


In [ ]:
display_query("""
SELECT mrn, full_name
FROM patients
WHERE full_name ILIKE 'c%';
""", title="Names beginning with C")


# Part IV · Missing Values: NULL

A missing database value is represented by `NULL`.

Do **not** write:

```sql
discharge_date = NULL
```

Use:

```sql
discharge_date IS NULL
```

### Why?

`NULL` means the value is unknown or missing. It is not an ordinary value that can be compared with `=`.


## 11. Find admissions without a discharge date


In [ ]:
display_query("""
SELECT
    admission_id,
    patient_id,
    diagnosis,
    admission_date,
    discharge_date
FROM admissions
WHERE discharge_date IS NULL
ORDER BY admission_date;
""", title="Currently open admissions")


# Part V · Sorting, Limiting, and DISTINCT

These clauses shape the final result without changing the stored data.

### ORDER BY

```sql
ORDER BY admission_date DESC
```

### LIMIT

```sql
LIMIT 3
```

### DISTINCT

```sql
SELECT DISTINCT gender
FROM patients;
```


## 12. Return the three most recent admissions


In [ ]:
display_query("""
SELECT admission_id, patient_id, diagnosis, admission_date
FROM admissions
ORDER BY admission_date DESC, admission_id DESC
LIMIT 3;
""", title="Three most recent admissions")


## 13. See unique values with DISTINCT


In [ ]:
display_query("""
SELECT DISTINCT gender
FROM patients
ORDER BY gender;
""", title="Distinct gender values")


# Part VI · How a SELECT Query Fits Together

A useful writing pattern is:

```sql
SELECT ...
FROM ...
WHERE ...
ORDER BY ...
LIMIT ...;
```

A simplified logical flow is:

```text
FROM
 ↓
WHERE
 ↓
SELECT
 ↓
ORDER BY
 ↓
LIMIT
```

This helps explain why aliases created in `SELECT` are not normally available inside the same query's `WHERE` clause.


## 14. Build one complete query

Question:

> Among active patients age 30 or older, show the three oldest patients with a clear output name.


In [ ]:
display_query("""
SELECT
    mrn,
    full_name AS patient,
    age
FROM patients
WHERE is_active = TRUE
  AND age >= 30
ORDER BY age DESC
LIMIT 3;
""", title="Complete SELECT pattern")


# Part VII · Changing Data Safely

Reading data is low risk. Changing data requires more discipline.

The three core statements are:

| Statement | Purpose |
|---|---|
| `INSERT` | add rows |
| `UPDATE` | change existing rows |
| `DELETE` | remove rows |

### The safety workflow

```text
1. Preview with SELECT
2. Verify the WHERE condition
3. Make the change
4. Inspect changed rows with RETURNING
5. COMMIT only when you intend to keep it
```

For teaching examples, we will use **ROLLBACK** so the original dataset stays intact.


## 15. A notebook helper for safe write practice

The next Python helper runs **one SQL write statement inside a transaction and always rolls it back**.

This is only a teaching convenience. The SQL statement itself is still ordinary PostgreSQL.


In [ ]:
def run_and_rollback(sql, title="Transaction result"):
    """Run one SQL statement, display RETURNING rows, then always roll it back."""
    if not db.connected:
        raise RuntimeError("Connect to PostgreSQL first.")

    with db.engine.connect() as conn:
        transaction = conn.begin()

        try:
            result = conn.execute(text(sql))

            if result.returns_rows:
                df = pd.DataFrame(result.fetchall(), columns=result.keys())
                print(title)
                display(df)

        finally:
            transaction.rollback()

    print("↩ Rolled back. The teaching database was not permanently changed.")


# Part VIII · INSERT

`INSERT` adds a new row.

A good pattern is:

```sql
INSERT INTO table_name (column1, column2, ...)
VALUES (...)
RETURNING ...;
```

Why list the columns explicitly?

Because it makes the statement easier to read and less dependent on the physical column order.


## 16. Insert a patient, inspect the row, then roll it back


In [ ]:
run_and_rollback("""
INSERT INTO patients (mrn, full_name, age, gender)
VALUES ('MRN1010', 'Ivy Carter', 50, 'Female')
RETURNING patient_id, mrn, full_name, age, gender;
""", title="Inserted row")


## Verify that the INSERT was rolled back


In [ ]:
display_query("""
SELECT patient_id, mrn, full_name
FROM patients
WHERE mrn = 'MRN1010';
""", title="MRN1010 after rollback")


# Part IX · UPDATE

An `UPDATE` changes existing rows.

The most important part is often the `WHERE` clause.

### Unsafe

```sql
UPDATE patients
SET age = 35;
```

That changes **every patient**.

### Safer

```sql
UPDATE patients
SET age = 35
WHERE mrn = 'MRN1003';
```

Use a stable identifier whenever possible.


## 17. Step 1 — preview the row before updating


In [ ]:
display_query("""
SELECT patient_id, mrn, full_name, age
FROM patients
WHERE mrn = 'MRN1003';
""", title="Preview UPDATE target")


## 18. Step 2 — test the UPDATE and roll it back


In [ ]:
run_and_rollback("""
UPDATE patients
SET age = 35
WHERE mrn = 'MRN1003'
RETURNING patient_id, mrn, full_name, age;
""", title="Updated row inside the transaction")


## 19. Step 3 — verify the original value is still present


In [ ]:
display_query("""
SELECT patient_id, mrn, full_name, age
FROM patients
WHERE mrn = 'MRN1003';
""", title="Value after rollback")


# Part X · DELETE and Constraints

`DELETE` removes rows, but database relationships can protect important data.

The `admissions.patient_id` foreign key uses `ON DELETE RESTRICT`.

That means PostgreSQL should reject an attempt to delete a patient who is still referenced by an admission.


## 20. Expected failure: try to delete a referenced patient

This cell intentionally demonstrates a constraint protecting the database.


In [ ]:
try:
    with db.engine.connect() as conn:
        tx = conn.begin()
        try:
            conn.execute(text("""
                DELETE FROM patients
                WHERE mrn = 'MRN1004';
            """))
            print("Unexpected: delete succeeded.")
        except Exception as e:
            print("✅ PostgreSQL blocked the delete as expected.")
            print("Reason: the patient is referenced by another table.")
            print(str(e).split("\n")[0])
        finally:
            tx.rollback()
except Exception as e:
    print(e)


## 21. A safe DELETE example with a disposable row

We can insert a temporary practice row and delete it inside the same transaction. Then we roll back the entire transaction.


In [ ]:
with db.engine.connect() as conn:
    tx = conn.begin()

    try:
        conn.execute(text("""
            INSERT INTO patients (mrn, full_name, age, gender)
            VALUES ('MRN1099', 'Practice Patient', 40, 'Other');
        """))

        result = conn.execute(text("""
            DELETE FROM patients
            WHERE mrn = 'MRN1099'
            RETURNING patient_id, mrn, full_name;
        """))

        df = pd.DataFrame(result.fetchall(), columns=result.keys())
        display(df)

    finally:
        tx.rollback()

print("↩ Entire practice transaction rolled back.")


# Part XI · Transactions

A transaction groups statements into one logical unit of work.

```sql
BEGIN;

-- one or more changes

COMMIT;
```

keeps the changes.

```sql
BEGIN;

-- one or more changes

ROLLBACK;
```

undoes the uncommitted changes.

### A practical interpretation of ACID

| Idea | Practical meaning |
|---|---|
| Atomicity | the unit succeeds or is undone |
| Consistency | database rules remain enforced |
| Isolation | concurrent work should not interfere incorrectly |
| Durability | committed changes survive |


## 22. Practice the transaction idea explicitly

We will update a row, inspect it **inside the transaction**, then roll back and verify it again.


In [ ]:
with db.engine.connect() as conn:
    tx = conn.begin()

    before = conn.execute(text("""
        SELECT mrn, full_name, age
        FROM patients
        WHERE mrn = 'MRN1005';
    """)).mappings().all()

    changed = conn.execute(text("""
        UPDATE patients
        SET age = 59
        WHERE mrn = 'MRN1005'
        RETURNING mrn, full_name, age;
    """)).mappings().all()

    print("Before:")
    display(pd.DataFrame(before))

    print("Inside the transaction after UPDATE:")
    display(pd.DataFrame(changed))

    tx.rollback()

print("After ROLLBACK:")
display_query("""
SELECT mrn, full_name, age
FROM patients
WHERE mrn = 'MRN1005';
""")


# Part XII · JSONB

PostgreSQL can store structured JSON documents in a `JSONB` column.

A sample note looks conceptually like:

```json
{
  "symptom": "wheezing",
  "severity": "moderate",
  "follow_up": true
}
```

### Two important operators

| Operator | Returns |
|---|---|
| `->` | JSON / JSONB |
| `->>` | text |

For most simple display and filtering tasks, `->>` is convenient.


## 23. Extract fields from JSONB


In [ ]:
display_query("""
SELECT
    patient_id,
    note->>'symptom' AS symptom,
    note->>'severity' AS severity,
    (note->>'follow_up')::boolean AS follow_up
FROM clinical_notes
ORDER BY patient_id;
""", title="JSONB fields as normal query columns")


## 24. Filter using a JSONB field


In [ ]:
display_query("""
SELECT
    patient_id,
    note->>'symptom' AS symptom,
    note->>'severity' AS severity
FROM clinical_notes
WHERE note->>'severity' IN ('moderate', 'severe')
ORDER BY patient_id;
""", title="Moderate or severe notes")


# Part XIII · Build a Table from Query Results

Sometimes a query result becomes the starting point for another workflow.

A simple pattern is:

```sql
CREATE TABLE ... AS
SELECT ...;
```

For this demonstration, we will create the table, inspect it, and then remove it so the notebook remains repeatable.


## 25. Create a senior-patient table from a SELECT


In [ ]:
execute_script("""
DROP TABLE IF EXISTS senior_patients;

CREATE TABLE senior_patients AS
SELECT
    patient_id,
    mrn,
    full_name,
    age
FROM patients
WHERE age >= 65;
""")

display_query("""
SELECT *
FROM senior_patients
ORDER BY age DESC;
""", title="senior_patients")


## Clean up the demonstration table


In [ ]:
execute_query("DROP TABLE IF EXISTS senior_patients;")
print("senior_patients removed.")


# Part XIV · MERGE — Optional Advanced Preview

`MERGE` is useful when incoming data may contain both:

- rows that already exist and should be updated,
- rows that do not exist and should be inserted.

Conceptually:

```text
incoming row
    ↓
does MRN already exist?
   / \
 yes  no
  ↓    ↓
UPDATE INSERT
```

This is sometimes called an **upsert-style** workflow.

PostgreSQL supports `MERGE` in PostgreSQL 15 and newer.


## 26. Check your PostgreSQL version


In [ ]:
version_df = execute_query("SHOW server_version_num;")
display(version_df)

server_version_num = int(version_df.iloc[0, 0])
print("MERGE available:", server_version_num >= 150000)


## 27. MERGE demonstration with automatic rollback

The entire demonstration occurs inside one transaction, so the patient table returns to its original state afterward.


In [ ]:
if server_version_num < 150000:
    print("This PostgreSQL version does not support MERGE. Skip this optional example.")
else:
    with db.engine.connect() as conn:
        tx = conn.begin()

        try:
            conn.execute(text("""
                CREATE TEMP TABLE incoming_patients (
                    mrn text,
                    full_name text,
                    age smallint,
                    gender text
                );
            """))

            conn.execute(text("""
                INSERT INTO incoming_patients VALUES
                    ('MRN1002', 'Bob Smith', 69, 'Male'),
                    ('MRN1007', 'Grace Hall', 55, 'Female');
            """))

            conn.execute(text("""
                MERGE INTO patients AS target
                USING incoming_patients AS source
                ON target.mrn = source.mrn

                WHEN MATCHED THEN
                    UPDATE SET
                        full_name = source.full_name,
                        age = source.age,
                        gender = source.gender

                WHEN NOT MATCHED THEN
                    INSERT (mrn, full_name, age, gender)
                    VALUES (
                        source.mrn,
                        source.full_name,
                        source.age,
                        source.gender
                    );
            """))

            result = conn.execute(text("""
                SELECT mrn, full_name, age, gender
                FROM patients
                WHERE mrn IN ('MRN1002', 'MRN1007')
                ORDER BY mrn;
            """))

            display(pd.DataFrame(result.fetchall(), columns=result.keys()))

        finally:
            tx.rollback()

    print("↩ MERGE demonstration rolled back.")


# Part XV · Common Mistakes

### 1. Using `= NULL`

Wrong:

```sql
WHERE discharge_date = NULL
```

Use:

```sql
WHERE discharge_date IS NULL
```

### 2. Forgetting `WHERE` in UPDATE or DELETE

Always preview the target rows first.

### 3. Filtering on the wrong data type

Text values need quotes:

```sql
WHERE gender = 'Female'
```

Numbers usually do not:

```sql
WHERE age >= 50
```

### 4. Assuming a write succeeded because there was no error

Use `RETURNING` or a follow-up `SELECT`.

### 5. Keeping a test change accidentally

Use a transaction and `ROLLBACK` while experimenting.


# Part XVI · Guided Practice

Try each task before opening its hint.


## Practice 1 · SELECT and WHERE

Return:

- `mrn`
- `full_name`
- `age`

for active patients age 50 or older. Sort oldest first.

<details>
<summary>Hint</summary>

Use:

```sql
WHERE is_active = TRUE
  AND age >= 50
```

</details>


In [ ]:
practice_1 = """
SELECT
    -- columns
FROM patients
WHERE
    -- conditions
ORDER BY
    -- sorting
;
"""

# display_query(practice_1, title="Practice 1")


## Practice 2 · Pattern matching

Find patients whose name begins with the letter C.

<details>
<summary>Hint</summary>

Try `ILIKE 'c%'`.

</details>


In [ ]:
practice_2 = """
SELECT
FROM patients
WHERE
;
"""


## Practice 3 · INSERT with RETURNING

Write an `INSERT` for a fictional patient and return the new row.

Do not permanently keep the row.

<details>
<summary>Hint</summary>

After writing the SQL string, you can test it with:

```python
run_and_rollback(your_sql)
```

</details>


In [ ]:
practice_3 = """
INSERT INTO patients (
    -- columns
)
VALUES (
    -- values
)
RETURNING
    -- columns to inspect
;
"""

# run_and_rollback(practice_3)


## Practice 4 · Safe UPDATE

Change `MRN1005` to age 59, but roll the change back.

Before the update, preview the row.


In [ ]:
# Step 1: preview
preview_query = """
SELECT
FROM patients
WHERE
;
"""

# Step 2: write the UPDATE
update_query = """
UPDATE patients
SET
WHERE
RETURNING
;
"""

# display_query(preview_query)
# run_and_rollback(update_query)


## Practice 5 · JSONB

Return:

- `patient_id`
- symptom
- severity

from `clinical_notes`.

Keep only notes where severity is moderate or severe.


In [ ]:
practice_5 = """
SELECT
    patient_id,
    -- extract symptom,
    -- extract severity
FROM clinical_notes
WHERE
    -- JSONB filter
ORDER BY patient_id;
"""

# display_query(practice_5, title="Practice 5")


## Practice 6 · Explain MERGE

In your own words, answer:

> When would `MERGE` be more useful than using only a simple `INSERT`?

<details>
<summary>Suggested answer</summary>

`MERGE` is useful when incoming rows can require different actions. Existing matches may need an update, while new rows may need an insert.

</details>


# Part XVII · Final Mini-Lab

Imagine you receive this request:

> “Show the three oldest active patients who are at least 30 years old. Then test changing one patient's age, verify the changed row, and undo the test.”

Break the task into two parts.

### Read task

1. Select the needed columns.
2. Filter active patients age 30+.
3. Sort oldest first.
4. Limit to three rows.

### Write task

1. Preview one patient using MRN.
2. Start/test the update.
3. Use `RETURNING`.
4. Roll it back.
5. Verify the original value.


In [ ]:
# Part A: write the SELECT query
final_read_query = """
SELECT
    -- columns
FROM patients
WHERE
    -- conditions
ORDER BY
    -- sort
LIMIT 3;
"""

# display_query(final_read_query, title="Final mini-lab: read task")


In [ ]:
# Part B: write the UPDATE statement
final_update_query = """
UPDATE patients
SET
    -- change
WHERE
    -- stable identifier
RETURNING
    -- columns to verify
;
"""

# run_and_rollback(final_update_query, title="Final mini-lab: write task")


# Quick Concept Check

### What clause filters rows?
<details><summary>Show answer</summary>`WHERE`</details>

### How do you test for a missing value?
<details><summary>Show answer</summary>`IS NULL` or `IS NOT NULL`</details>

### What does `RETURNING` help you do?
<details><summary>Show answer</summary>Inspect rows affected by an INSERT, UPDATE, or DELETE.</details>

### What is the difference between COMMIT and ROLLBACK?
<details><summary>Show answer</summary>COMMIT keeps the transaction's changes; ROLLBACK undoes its uncommitted changes.</details>

### Why preview rows before UPDATE or DELETE?
<details><summary>Show answer</summary>To verify that the WHERE condition targets exactly the intended rows.</details>

### What does `->>` do with JSONB?
<details><summary>Show answer</summary>It extracts a JSON field as text.</details>


# Module 3 Summary

The most important habit from this module is not a single SQL command.

It is this workflow:

```text
Read first
  ↓
Narrow the rows
  ↓
Inspect the result
  ↓
Change carefully
  ↓
Use RETURNING
  ↓
ROLLBACK while testing
  ↓
Verify
```

You should now be able to:

- write clear `SELECT` queries,
- filter and sort results,
- handle `NULL`,
- use aliases and calculated columns,
- safely practice `INSERT`, `UPDATE`, and `DELETE`,
- explain transactions,
- query JSONB,
- create a table from a query,
- explain when `MERGE` is useful.

## Next
**Module 4: Aggregating Data with PostgreSQL**
